# Clustering Places of Worship Features

This analysis clusters haunted locations based on their proximity to places of worship and the types of apparitions reported. The features used for clustering are:

- **Haunted_Place_Proximity**: Measures how close a haunted location is to a place of worship in meters.
- **Distance_to_Nearest_Worship**: A boolean indicating whether a haunted place is within a 5-mile radius of a worship site.
- **Religion_Intersection**: The dominant religious affiliation near the haunted place.
- **Apparition_Type**: The type of supernatural entity reported (e.g., ghost, demon, spirit).

By analyzing these factors, we aim to explore whether religious environments influence the types of supernatural phenomena reported and how belief systems may shape paranormal experiences.

# Generating Indices

The indices were chosen to ensure a balanced and representative sample of haunted places based on proximity to places of worship, religious affiliation, and apparition type. First, 500 haunted places closest to worship centers and 500 farthest away were selected to compare supernatural activity in religious vs. secular areas. Then, a diverse sample of up to 200 haunted places near different religious sites (Christian vs Muslim vs Jewish...etc) was included to analyze how religious beliefs may influence reported apparitions. Finally, hauntings were grouped by apparition type, with up to 100 samples per type randomly selected to prevent any one category from dominating. The final dataset combines these selections, allowing for meaningful clustering that examines the relationship between supernatural reports, religious proximity, and belief systems.

In [ ]:
# Convert relevant columns to numeric and boolean types
df['Haunted_Place_Proximity'] = pd.to_numeric(df['Haunted_Place_Proximity'], errors='coerce')
df['Distance_to_Nearest_Worship'] = df['Distance_to_Nearest_Worship'].astype(bool)

# Number of rows to select per extreme group (adjust as needed)
num_rows_per_group = 500  

# Select haunted places CLOSEST to places of worship
close_worship = df[df['Distance_to_Nearest_Worship'] == True].nsmallest(num_rows_per_group, 'Haunted_Place_Proximity')

# Select haunted places FARTHEST from places of worship
far_worship = df[df['Distance_to_Nearest_Worship'] == False].nlargest(num_rows_per_group, 'Haunted_Place_Proximity')

# Get the index numbers of these rows
close_worship_indices = close_worship.index.tolist()
far_worship_indices = far_worship.index.tolist()

# Combine both sets of indices
selected_indices = close_worship_indices + far_worship_indices

# List of religions to sample from
religions = ['christian', 'muslim', 'jewish', 'buddhist', 'hindu', 'shinto', 
             'pagan', 'spiritualist', 'multifaith']

# Select a balanced sample of haunted places near different religious sites
religion_samples = [
    df[df['Religion_Intersection'] == r].sample(n=min(200, len(df[df['Religion_Intersection'] == r])), random_state=42)
    for r in religions if not df[df['Religion_Intersection'] == r].empty
]

# Get the index numbers of religion-based samples
religion_indices = [sample.index.tolist() for sample in religion_samples]
religion_indices = [index for sublist in religion_indices for index in sublist]  # Flatten list

# Select a subset of hauntings by apparition type
selected_hauntings = df.loc[selected_indices].groupby('Apparition_Type').apply(lambda x: x.sample(n=min(100, len(x)), random_state=42))

# Get the indices of selected hauntings by apparition type
apparition_indices = selected_hauntings.index.get_level_values(1).tolist()

# Combine all selected indices
final_selected_indices = apparition_indices + religion_indices

# # Print the final indices
# print("Selected Haunted Place Indices:", final_selected_indices)

# Jaccard

In [73]:
import pandas as pd
import os
import sys 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/jaccard/religion/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Jaccard Clusters

In [76]:
# Haunted Place Proximity & Worship Distance Analysis
print("Haunted Place Proximity & Worship Distance by Cluster:")
print(df.groupby('Cluster')[['Haunted_Place_Proximity']].describe())

# Count places where a house of worship is nearby
print("\nProportion of Haunted Places Near Worship Centers by Cluster:")
print(df.groupby('Cluster')['Distance_to_Nearest_Worship'].value_counts(normalize=True))

# Apparition Type Distribution by Cluster
print("\nApparition Type Distribution Across Clusters:")
apparition_counts = df.groupby(['Cluster', 'Apparition_Type']).size().unstack(fill_value=0)
print(apparition_counts)

# Normalize apparition types within each cluster
print("\nNormalized Apparition Type Proportions by Cluster:")
apparition_proportions = apparition_counts.div(apparition_counts.sum(axis=1), axis=0)
print(apparition_proportions)

# Religion & Apparition Relationship
print("\nReligion Intersection Across Clusters:")
religion_counts = df.groupby(['Cluster', 'Religion_Intersection']).size().unstack(fill_value=0)
print(religion_counts)

Haunted Place Proximity & Worship Distance by Cluster:
          Haunted_Place_Proximity                   
                            count unique    top freq
Cluster                                             
cluster 0                      38      2  False   20
cluster 1                      57      2   True   39
cluster 2                     601      2   True  359

Proportion of Haunted Places Near Worship Centers by Cluster:
Cluster    Distance_to_Nearest_Worship
cluster 0  237.28                         0.026316
           13137.88                       0.026316
           9692.51                        0.026316
           9979.05                        0.026316
           10242.98                       0.026316
                                            ...   
cluster 2  1521.92                        0.001664
           1512.41                        0.001664
           1505.67                        0.001664
           1491.92                        0.001664
           7870

# Edit Distance

In [83]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/edit-distance/religion/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Edit Distance Clusters

In [86]:
# Haunted Place Proximity & Worship Distance Analysis
print("Haunted Place Proximity & Worship Distance by Cluster:")
print(df.groupby('Cluster')[['Haunted_Place_Proximity']].describe())

# Count places where a house of worship is nearby
print("\nProportion of Haunted Places Near Worship Centers by Cluster:")
print(df.groupby('Cluster')['Distance_to_Nearest_Worship'].value_counts(normalize=True))

# Apparition Type Distribution by Cluster
print("\nApparition Type Distribution Across Clusters:")
apparition_counts = df.groupby(['Cluster', 'Apparition_Type']).size().unstack(fill_value=0)
print(apparition_counts)

# Normalize apparition types within each cluster
print("\nNormalized Apparition Type Proportions by Cluster:")
apparition_proportions = apparition_counts.div(apparition_counts.sum(axis=1), axis=0)
print(apparition_proportions)

# Religion & Apparition Relationship
print("\nReligion Intersection Across Clusters:")
religion_counts = df.groupby(['Cluster', 'Religion_Intersection']).size().unstack(fill_value=0)
print(religion_counts)

Haunted Place Proximity & Worship Distance by Cluster:
          Haunted_Place_Proximity                  
                            count unique   top freq
Cluster                                            
cluster 0                       1      1  True    1
cluster 2                      27      2  True   24
cluster 3                     246      2  True  127
cluster 4                     276      2  True  168
cluster 5                     146      2  True   96

Proportion of Haunted Places Near Worship Centers by Cluster:
Cluster    Distance_to_Nearest_Worship
cluster 0  732.35                         1.000000
cluster 2  795.13                         0.111111
           1220.72                        0.037037
           147.14                         0.037037
           20537.22                       0.037037
                                            ...   
cluster 5  2450.10                        0.006849
           2443.05                        0.006849
           2295.86 

# Cosine Similarity

In [88]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

from dsci_550_a1.unpack_circles import unpack_cluster

haunted_places = "../data/processed/haunted_places_features_added.tab"

df = pd.read_csv(haunted_places, sep = "\t")

clusters = unpack_cluster("../clustering/cosine/religion/visualization/clusters.json")

cluster_labels = {}  # Create a dictionary to map row indices to cluster labels

for cluster_name, indices in clusters.items():
    for index in indices:
        cluster_labels[index] = cluster_name  # Assign cluster name

# Add the 'Cluster' column
df["Cluster"] = df.index.map(cluster_labels)

### Investigating Cosine Clusters

In [91]:
# Haunted Place Proximity & Worship Distance Analysis
print("Haunted Place Proximity & Worship Distance by Cluster:")
print(df.groupby('Cluster')[['Haunted_Place_Proximity']].describe())

# Count places where a house of worship is nearby
print("\nProportion of Haunted Places Near Worship Centers by Cluster:")
print(df.groupby('Cluster')['Distance_to_Nearest_Worship'].value_counts(normalize=True))

# Apparition Type Distribution by Cluster
print("\nApparition Type Distribution Across Clusters:")
apparition_counts = df.groupby(['Cluster', 'Apparition_Type']).size().unstack(fill_value=0)
print(apparition_counts)

# Normalize apparition types within each cluster
print("\nNormalized Apparition Type Proportions by Cluster:")
apparition_proportions = apparition_counts.div(apparition_counts.sum(axis=1), axis=0)
print(apparition_proportions)

# Religion & Apparition Relationship
print("\nReligion Intersection Across Clusters:")
religion_counts = df.groupby(['Cluster', 'Religion_Intersection']).size().unstack(fill_value=0)
print(religion_counts)

Haunted Place Proximity & Worship Distance by Cluster:
          Haunted_Place_Proximity                  
                            count unique   top freq
Cluster                                            
cluster 0                     696      2  True  416

Proportion of Haunted Places Near Worship Centers by Cluster:
Cluster    Distance_to_Nearest_Worship
cluster 0  0.00                           0.017241
           795.13                         0.015805
           157.40                         0.007184
           2327.27                        0.007184
           845.79                         0.005747
                                            ...   
           1469.11                        0.001437
           1477.56                        0.001437
           1487.48                        0.001437
           1491.92                        0.001437
           787029.86                      0.001437
Name: proportion, Length: 639, dtype: float64

Apparition Type Distributio